In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gc
import sys
from dotenv import load_dotenv

load_dotenv()
os.chdir(os.getenv("ROOT"))
sys.path.insert(0, os.getenv("ROOT"))

from src.experiment_notebook import setup_experiment, fine_tune_and_evaluate

In [ ]:
# ── Configuració del sweep ──────────────────────────────────────────────────
RESULTS_FILE   = 'sweep_results_v2.json'   # fitxer on es guarden tots els runs
N_RUNS         = 3                          # models per configuració
BASE_SEED      = 42                         # seeds: BASE_SEED, BASE_SEED+1, …

ALPHAS       = [0.0, 0.1, 0.5, 1.0, 5.0]
LAYER_SIZES  = [32, 64, 128, 256]

EPOCHS_PER_ITER = 5
NUM_ITERATIONS  = 10
BATCH_SIZE      = 32

In [ ]:
# ── Sweep principal ─────────────────────────────────────────────────────────
# Cada entrada de `all_results` representa UN run individual.
# Camps: alpha, ls, run_id, seed, accuracy, tpr_gap, tpr_male, tpr_female,
#        auc_all, gender_auc_last_layer

# Carrega resultats previs si existeix el fitxer (permet reprendre el sweep)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    print(f"Carregats {len(all_results)} runs previs de '{RESULTS_FILE}'")
else:
    all_results = []
    print("Fitxer de resultats nou.")

# Conjunt de (alpha, ls, run_id) ja completats
done = {(r['alpha'], r['ls'], r['run_id']) for r in all_results}

total_configs = len(ALPHAS) * len(LAYER_SIZES) * N_RUNS
remaining     = total_configs - len(done)
print(f"Total configuracions: {total_configs}  |  Pendents: {remaining}")

for alpha in ALPHAS:
    for ls in LAYER_SIZES:
        for run_id in range(N_RUNS):
            if (alpha, ls, run_id) in done:
                print(f"[SKIP] alpha={alpha}, ls={ls}, run={run_id}")
                continue

            seed = BASE_SEED + run_id
            print(f"\n{'='*60}")
            print(f"alpha={alpha}, ls={ls}, run={run_id}/{N_RUNS-1}, seed={seed}")
            print('='*60)

            try:
                ctx = setup_experiment(
                    alpha=alpha,
                    prev2last_layer_size=ls,
                    batch_size=BATCH_SIZE,
                    seed=seed,
                    run_name=f"sweep_a{alpha}_ls{ls}_r{run_id}"
                )

                result = None
                for it in range(NUM_ITERATIONS):
                    print(f"  Iteration {it+1}/{NUM_ITERATIONS}")
                    result = fine_tune_and_evaluate(
                        ctx, epochs=EPOCHS_PER_ITER, alpha=alpha
                    )

                # Extreure mètriques del resultat final de la iteració
                cm_all = result['All']['confusion_matrix']
                tn_a, fp_a, fn_a, tp_a = cm_all[0][0], cm_all[0][1], cm_all[1][0], cm_all[1][1]

                cm_m = result['Male']['confusion_matrix']
                tpr_male = cm_m[1][1] / (cm_m[1][1] + cm_m[1][0]) if (cm_m[1][1] + cm_m[1][0]) > 0 else float('nan')

                cm_f = result['Female']['confusion_matrix']
                tpr_female = cm_f[1][1] / (cm_f[1][1] + cm_f[1][0]) if (cm_f[1][1] + cm_f[1][0]) > 0 else float('nan')

                tpr_gap = abs(tpr_male - tpr_female)

                # Gender AUC de l'última capa
                gender_pred = result.get('gender_pred', {})
                last_layer_key = sorted(
                    [k for k in gender_pred.keys() if k.startswith('layer_')],
                    key=lambda x: int(x.split('_')[1])
                )[-1] if gender_pred else None
                gender_auc_last = gender_pred[last_layer_key]['auc'] if last_layer_key else float('nan')

                run_result = {
                    'alpha'               : alpha,
                    'ls'                  : ls,
                    'run_id'              : run_id,
                    'seed'                : seed,
                    'accuracy'            : result['All']['accuracy'],
                    'auc_all'             : result['All']['auc'],
                    'tpr_male'            : tpr_male,
                    'tpr_female'          : tpr_female,
                    'tpr_gap'             : tpr_gap,
                    'gender_auc_last_layer': gender_auc_last,
                    'results_dir'         : ctx['results_dir'],
                }
                all_results.append(run_result)
                done.add((alpha, ls, run_id))

                # Guardar incrementalment
                with open(RESULTS_FILE, 'w') as f:
                    json.dump(all_results, f, indent=2)
                print(f"  Run guardat a '{RESULTS_FILE}'")

            except Exception as e:
                print(f"  ERROR: {e}")
                import traceback; traceback.print_exc()

            finally:
                # Alliberar memòria GPU entre runs
                if 'ctx' in dir():
                    del ctx
                gc.collect()
                torch.cuda.empty_cache()

print("\nSweep completat!")

In [ ]:
# ── Carregar resultats i calcular mitjana ± std per configuració ────────────
with open(RESULTS_FILE, 'r') as f:
    all_results = json.load(f)

df_runs = pd.DataFrame(all_results)

# Mètriques a agregar
METRICS = ['accuracy', 'auc_all', 'tpr_male', 'tpr_female', 'tpr_gap', 'gender_auc_last_layer']

agg = df_runs.groupby(['alpha', 'ls'])[METRICS].agg(['mean', 'std']).reset_index()
agg.columns = ['alpha', 'ls'] + [f'{m}_{s}' for m, s in agg.columns[2:]]

# Nombre de runs per configuració
agg['n_runs'] = df_runs.groupby(['alpha', 'ls']).size().values

agg = agg.sort_values(['alpha', 'ls']).reset_index(drop=True)

# Taula: mitjana ± std
display_df = agg[['alpha', 'ls', 'n_runs']].copy()
for m in METRICS:
    display_df[m] = agg[f'{m}_mean'].map('{:.3f}'.format) + ' ± ' + agg[f'{m}_std'].map('{:.3f}'.format)

display(display_df.style
        .set_caption(f'Resultats sweep (mitjana ± std, N={N_RUNS} runs per config)')
)

In [ ]:
# ── Heatmaps: accuracy i tpr_gap (mitjana) amb barres d'error (std) ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, cmap, title in [
    (axes[0], 'accuracy', 'Greens', 'Accuracy (mitjana)'),
    (axes[1], 'tpr_gap',  'Reds',   'TPR gap |male−female| (mitjana)'),
]:
    pivot_mean = agg.pivot(index='alpha', columns='ls', values=f'{metric}_mean')
    pivot_std  = agg.pivot(index='alpha', columns='ls', values=f'{metric}_std')

    im = ax.imshow(pivot_mean.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(pivot_mean.columns)))
    ax.set_xticklabels(pivot_mean.columns)
    ax.set_yticks(range(len(pivot_mean.index)))
    ax.set_yticklabels(pivot_mean.index)
    ax.set_xlabel('Layer Size')
    ax.set_ylabel('Alpha')
    ax.set_title(title)
    for i in range(len(pivot_mean.index)):
        for j in range(len(pivot_mean.columns)):
            m_val = pivot_mean.values[i, j]
            s_val = pivot_std.values[i, j]
            if not np.isnan(m_val):
                ax.text(j, i, f'{m_val:.3f}\n±{s_val:.3f}',
                        ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax)

plt.suptitle('Sweep bias–accuracy (mitjana ± std)', fontsize=13)
plt.tight_layout()
plt.savefig('sweep_heatmaps_v2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Gender AUC: heatmap + corbes amb barres d'error + scatter ──────────────
df_gender = agg.dropna(subset=['gender_auc_last_layer_mean']).copy()

if df_gender.empty:
    print("No hi ha dades de 'gender_auc_last_layer' als resultats.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # --- 1. Heatmap: Gender AUC (mitjana) ---
    pivot_gauc_m = df_gender.pivot(index='alpha', columns='ls', values='gender_auc_last_layer_mean')
    pivot_gauc_s = df_gender.pivot(index='alpha', columns='ls', values='gender_auc_last_layer_std')
    im = axes[0].imshow(pivot_gauc_m.values, cmap='RdYlGn_r', aspect='auto', vmin=0.5, vmax=1.0)
    axes[0].set_xticks(range(len(pivot_gauc_m.columns)))
    axes[0].set_xticklabels(pivot_gauc_m.columns)
    axes[0].set_yticks(range(len(pivot_gauc_m.index)))
    axes[0].set_yticklabels(pivot_gauc_m.index)
    axes[0].set_xlabel('Layer Size')
    axes[0].set_ylabel('Alpha')
    axes[0].set_title('Gender AUC última capa (mitjana)')
    for i in range(len(pivot_gauc_m.index)):
        for j in range(len(pivot_gauc_m.columns)):
            v, s = pivot_gauc_m.values[i, j], pivot_gauc_s.values[i, j]
            if not np.isnan(v):
                axes[0].text(j, i, f'{v:.3f}\n±{s:.3f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=axes[0], label='AUC (0.5=ideal)')

    # --- 2. Corbes amb barres d'error: Gender AUC vs Alpha ---
    for ls in sorted(df_gender['ls'].unique()):
        sub = df_gender[df_gender['ls'] == ls].sort_values('alpha')
        axes[1].errorbar(
            sub['alpha'],
            sub['gender_auc_last_layer_mean'],
            yerr=sub['gender_auc_last_layer_std'],
            marker='o', capsize=4, label=f'ls={ls}'
        )
    axes[1].axhline(0.5, color='green', linestyle='--', alpha=0.6, label='AUC=0.5 (ideal)')
    axes[1].set_xlabel('Alpha')
    axes[1].set_ylabel('Gender AUC (última capa)')
    axes[1].set_title('Presència del sexe vs Alpha (±std)')
    axes[1].set_ylim(0.45, 1.05)
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # --- 3. Scatter: Bias vs Gender AUC, amb barres d'error ---
    alpha_colors = {0.0: '#1f77b4', 0.1: '#ff7f0e', 0.25: '#ffbb78', 0.5: '#2ca02c',
                    0.75: '#98df8a', 1.0: '#d62728', 5.0: '#9467bd', 10.0: '#8c564b'}
    ls_markers   = {16: 'v', 32: 'o', 48: 'p', 64: 's', 128: '^', 256: 'D', 512: 'P'}

    for _, row in df_gender.iterrows():
        a, l = row['alpha'], row['ls']
        axes[2].errorbar(
            row['gender_auc_last_layer_mean'], row['tpr_gap_mean'],
            xerr=row['gender_auc_last_layer_std'], yerr=row['tpr_gap_std'],
            fmt=ls_markers.get(l, 'x'),
            color=alpha_colors.get(a, 'gray'),
            markersize=9, capsize=3,
            markeredgecolor='black', markeredgewidth=0.5
        )
    axes[2].axvline(0.5, color='green', linestyle='--', alpha=0.6, label='Gender AUC=0.5 (ideal)')
    axes[2].set_xlabel("Gender AUC última capa")
    axes[2].set_ylabel('Bias (|TPR gap|)')
    axes[2].set_title('Relació Presència Sexe — Bias (±std)')
    axes[2].grid(True, alpha=0.3)

    from matplotlib.lines import Line2D
    legend_alpha = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                           markersize=9, label=f'α={a}') for a, c in alpha_colors.items()]
    legend_ls    = [Line2D([0],[0], marker=m, color='w', markerfacecolor='gray',
                           markersize=9, label=f'ls={l}') for l, m in ls_markers.items()]
    leg1 = axes[2].legend(handles=legend_alpha, title='Alpha', loc='upper left',
                          bbox_to_anchor=(1.02, 1), fontsize=8)
    axes[2].add_artist(leg1)
    axes[2].legend(handles=legend_ls, title='Layer Size', loc='upper left',
                   bbox_to_anchor=(1.02, 0.5), fontsize=8)

    plt.suptitle('Presència del sexe a l\'última capa — runs múltiples', fontsize=14)
    plt.tight_layout()
    plt.savefig('gender_presence_last_layer_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Gràfic guardat en gender_presence_last_layer_v2.png")

In [ ]:
# ── Distribució individual dels runs (boxplots) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df_runs_loaded = pd.DataFrame(all_results)
df_runs_loaded['config'] = df_runs_loaded.apply(
    lambda r: f"α={r['alpha']}\nls={r['ls']}", axis=1
)
configs = (df_runs_loaded
           .sort_values(['alpha', 'ls'])
           .drop_duplicates('config')['config']
           .tolist())

for ax, metric, title in [
    (axes[0], 'accuracy', 'Accuracy per configuració'),
    (axes[1], 'tpr_gap',  'TPR gap per configuració'),
]:
    data_per_config = [
        df_runs_loaded[df_runs_loaded['config'] == c][metric].dropna().tolist()
        for c in configs
    ]
    ax.boxplot(data_per_config, labels=configs, patch_artist=True)
    ax.set_xlabel('Configuració (alpha, layer size)')
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.tick_params(axis='x', labelsize=6, rotation=45)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Distribució dels {N_RUNS} runs per configuració', fontsize=13)
plt.tight_layout()
plt.savefig('sweep_boxplots_v2.png', dpi=150, bbox_inches='tight')
plt.show()